# AutoML erweitert: neue Merkmale und Mais/Soja-Spezialist

**Autor:** jd · **Datum:** 2026-10-29 · **Issue:** –

**Ziel:** Die Notebooks 15–19 haben jede Idee einzeln gegen die Referenz geprüft – keine hat
geholfen. Hier sucht Optuna, ob eine **Kombination** etwas bringt: Trockenmasse-Indizes,
Boden-Indizes und Ableitungsspektren werden zu Schaltern im Suchraum, dazu optional der
Mais/Soja-Spezialist (`AutoMLConfig(extra_features=True, specialist=True)`). Stacking bleibt
draußen (ein Versuch dauert Minuten, einzeln klar schlechter).

Kombinierte Klasse, SVM/Extra Trees/MLP, 1 h. Validiert wird **nur**, wenn der beste Versuch die
Referenz (`combined_svm_aez_month`, CV 0.864) um mindestens `MIN_CV_GAIN` (0.005) schlägt.

**Aufbau:** Teil A erzeugt die Läufe (MLflow), Teil B belegt jede Erkenntnis mit einem Plot.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

from awp2.automl import AutoMLConfig, automl
from awp2.config import (
    AUTOML_APPROACH,
    EXTRA_TREES_FAMILY,
    MIN_CV_GAIN,
    MLP_FAMILY,
    SVM_FAMILY,
)
from awp2.data import load_split
from awp2.evaluation import compare_predictions
from awp2.experiment import RunConfig, out_of_fold_predictions, run
from awp2.models.combined import make_combined_svm
from awp2.plots import plot_cv_scores, plot_metric_comparison
from awp2.preprocessing import PreprocessingConfig

# A · Läufe

In [ ]:
reference_model = make_combined_svm().set_params(estimator__C=300.0, estimator__gamma=0.003)
reference_config = RunConfig(
    name="combined_svm_aez_month",
    approach="combined",
    preprocessing=PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True),
    track=False,
)
REFERENCE_CV = 0.864

### Suche

Höchstens 300 Versuche oder 1 h, Stand in `models/automl.db`.

In [ ]:
search = automl(
    AutoMLConfig(
        name="automl_extended",
        description="Optuna über SVM/Extra Trees/MLP, Vorverarbeitung inkl. Trockenmasse-, "
        "Boden-Indizes und Ableitungsspektren, optional Mais/Soja-Spezialist.",
        families=(SVM_FAMILY, EXTRA_TREES_FAMILY, MLP_FAMILY),
        extra_features=True,
        specialist=True,
    )
)
print(f"Bester CV-Score: {search.best_score:.4f} (Referenz {REFERENCE_CV})")
search.best_params

### Gepaarter Vergleich auf dem Trainingsteil (out-of-fold)

In [ ]:
best_config = RunConfig(
    name="automl_extended_best",
    approach=AUTOML_APPROACH,
    preprocessing=search.best.preprocessing,
    balance_samples=search.best.balance_samples,
    track=False,
)
y_train = load_split().y_train
oof_reference = out_of_fold_predictions(reference_model, reference_config)
oof_best = out_of_fold_predictions(search.best.model, best_config)
compare_predictions(y_train, oof_reference, oof_best)

### Entscheidung: nur bei Gewinn über der Rauschschwelle auf die Validierung

In [ ]:
gain = search.best_score - REFERENCE_CV
print(f"CV-Gewinn {gain:+.4f} (Schwelle {MIN_CV_GAIN})")
if gain >= MIN_CV_GAIN:
    best = run(
        search.best.model,
        RunConfig(
            name="automl_extended_best",
            approach=AUTOML_APPROACH,
            description="Bester Versuch von automl_extended, einmal auf der Validierung.",
            preprocessing=search.best.preprocessing,
            balance_samples=search.best.balance_samples,
            tuning_run=search.run_id,
        ),
    )
    reference = run(reference_model, reference_config)
    display(
        plot_metric_comparison(
            {"Referenz-SVM": reference.metrics, "AutoML erweitert": best.metrics}
        )
    )
else:
    print("Kein Gewinn über der Rauschschwelle – keine Validierung, Referenz bleibt.")

# B · Erkenntnisse

## 1. Suchverlauf

In [ ]:
states = pd.Series([trial.state.name for trial in search.study.trials]).value_counts()
print(states.to_string())
ax = plot_optimization_history(search.study)
ax.axhline(REFERENCE_CV, color="black", linestyle="--", linewidth=1, label="Referenz-SVM")
ax.legend()

## 2. Bringen die neuen Schalter etwas?

Bester CV-Score mit und ohne jeden Schalter. Liegen „an“ und „aus“ gleichauf, trägt der Schalter
auch in Kombination nichts bei.

In [ ]:
params = pd.DataFrame(list(search.candidates["params"]))
switches = [
    "use_dry_matter_indices",
    "use_soil_indices",
    "use_spectral_derivative",
    "use_specialist",
]
groups = {
    f"{switch} = {'an' if value else 'aus'}": search.candidates[params[switch] == value]
    for switch in switches
    for value in (False, True)
    if (params[switch] == value).any()
}
ax = plot_cv_scores(groups, reference=REFERENCE_CV)
ax.set_title("Bester CV-Score mit und ohne jeden neuen Schalter (gestrichelt: Referenz)")

In [ ]:
top = params.loc[search.candidates.nlargest(20, "mean").index]
print("Anteil der 20 besten Versuche mit eingeschaltetem Schalter:")
print(top[switches].astype(bool).mean().round(2).to_string())
print()
print(top["family"].value_counts().to_string())

## 3. Welche Entscheidungen zählen

In [ ]:
plot_param_importances(search.study)

# Fazit

- 300 Versuche (155 abgeschlossen, 145 abgebrochen). Bester Versuch: RBF-SVM (`C=172`,
  `gamma=0.0042`), AEZ × Monat, Boden-Indizes; CV **0.866** gegen 0.864 (+0.002 < `MIN_CV_GAIN`).
  Gepaart out-of-fold kein Unterschied (32 : 24 Zeilen, p = 0.35) – **keine Validierung**.

| Schalter | bester CV an | bester CV aus | Anteil unter den 20 besten |
| --- | --- | --- | --- |
| Trockenmasse-Indizes | 0.863 | 0.866 | 0 % |
| Boden-Indizes | 0.866 | 0.866 | 65 % |
| Ableitungsspektren | 0.856 | 0.866 | 0 % |
| Mais/Soja-Spezialist | 0.858 | 0.866 | 0 % |

- Auch **in Kombination** trägt keiner der neuen Schalter: Trockenmasse, Ableitung und Spezialist
  kommen unter den 20 besten Versuchen nie vor; die Boden-Indizes sind oft dabei, ändern den
  Score aber nicht (an und aus gleichauf).
- Alle 20 besten Versuche sind SVMs – wie in den früheren Suchen.
- Bestätigt die Einzel-Notebooks 15–19: Die Referenz-SVM bleibt.